## Bringing Reddit AITA Stories to Life with Azure AI Text-to-Speech


In [ ]:
import os
from dotenv import load_dotenv
import azure.cognitiveservices.speech as speechsdk

# Load environment variables from .env file
load_dotenv('env.env')

def read_text_file(file_path, encoding='utf-8'):
    with open(file_path, 'r', encoding=encoding) as file:
        text = file.read()
    return text

def text_to_speech(speech_key, service_region, text):
    # Set up Azure text-to-speech
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)
    audio_config = speechsdk.audio.AudioOutputConfig(use_default_speaker=True)

    # Set the neural voice
    speech_config.speech_synthesis_voice_name = 'en-US-KaiNeural'
    speech_synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_config)

    # Synthesize the text
    result = speech_synthesizer.speak_text_async(text).get()

    # Check the result
    if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
        print("Speech synthesized to speaker for text [{}]".format(text))
    elif result.reason == speechsdk.ResultReason.Canceled:
        cancellation_details = result.cancellation_details
        print("Speech synthesis canceled: {}".format(cancellation_details.reason))
        if cancellation_details.reason == speechsdk.CancellationReason.Error:
            if cancellation_details.error_details:
                print("Error details: {}".format(cancellation_details.error_details))
        print("Did you update the subscription info?")

if __name__ == "__main__":
    # Load the API key and region from environment variables
    speech_key = os.getenv("AZURE_SPEECH_KEY")
    service_region = os.getenv("AZURE_SERVICE_REGION")
    file_path = os.getenv("FILE_PATH")

    missing = [k for k, v in [("AZURE_SPEECH_KEY", speech_key), ("AZURE_SERVICE_REGION", service_region), ("FILE_PATH", file_path)] if not v]
    if missing:
        raise EnvironmentError(f"Missing required environment variables: {', '.join(missing)}")

    text = read_text_file(file_path, encoding='utf-8')
    text_to_speech(speech_key, service_region, text)